# 04 Service frequency from GTFS
Turns the England GTFS timetable into **bus departures per hour by time band** for each LSOA.

Method: pick one representative weekday, keep the trips running that day, count departures per stop in each time band, assign stops to LSOAs (point in polygon), sum per LSOA.

The `stop_times` file is huge, so it is read in chunks. Expect several minutes. Run from the `notebooks/` folder.

In [13]:
from pathlib import Path
import zipfile, io
import numpy as np, pandas as pd, geopandas as gpd
RAW = Path("../data/raw"); CLEAN = Path("../data/clean")
GTFS = RAW / "gtfs_england.zip"
TARGET_DATE = "2026-10-06"          # a Tuesday inside the feed (change if the check below finds no trips)
BANDS = {"morning": (7, 10), "off_peak": (10, 16), "evening": (16, 19)}   # [start, end) hour
CHUNK = 5_000_000
print(GTFS.exists(), GTFS.stat().st_size / 1e9 if GTFS.exists() else "")

True 1.162992616


## 1. Which trips run on the target date?
Uses `calendar.txt` (weekly pattern and date range) and `calendar_dates.txt` (added/removed days).

In [14]:
z = zipfile.ZipFile(GTFS)
print([n for n in z.namelist()])
d = pd.Timestamp(TARGET_DATE); ds = d.strftime("%Y%m%d"); dow = d.day_name().lower()
cal = pd.read_csv(z.open("calendar.txt"), dtype={"service_id": str, "start_date": str, "end_date": str})
print("feed validity:", cal.start_date.min(), "to", cal.end_date.max())
base = cal[(cal[dow] == 1) & (cal.start_date <= ds) & (cal.end_date >= ds)].service_id
services = set(base)
if "calendar_dates.txt" in z.namelist():
    cd = pd.read_csv(z.open("calendar_dates.txt"), dtype={"service_id": str, "date": str})
    cd = cd[cd.date == ds]
    services |= set(cd[cd.exception_type == 1].service_id)
    services -= set(cd[cd.exception_type == 2].service_id)
print("services running on", TARGET_DATE, ":", len(services))
trips = pd.read_csv(z.open("trips.txt"), usecols=["trip_id", "service_id"], dtype=str)
active = set(trips[trips.service_id.isin(services)].trip_id)
print("active trips:", len(active))
assert len(active) > 0, "No trips on that date: change TARGET_DATE to a date inside the feed validity"

['agency.txt', 'stops.txt', 'routes.txt', 'calendar.txt', 'calendar_dates.txt', 'trips.txt', 'shapes.txt', 'frequencies.txt', 'feed_info.txt', 'stop_times.txt']
feed validity: 20260930 to 20271201
services running on 2026-10-06 : 1037
active trips: 384291


## 2. Count departures per stop and time band (chunked)

In [15]:
counts = {b: {} for b in BANDS}
rows_read = 0
for chunk in pd.read_csv(z.open("stop_times.txt"), usecols=["trip_id", "stop_id", "departure_time"],
                         dtype=str, chunksize=CHUNK):
    rows_read += len(chunk)
    chunk = chunk[chunk.trip_id.isin(active)]
    hour = pd.to_numeric(chunk.departure_time.str.slice(0, 2), errors="coerce") % 24
    for b, (a, e) in BANDS.items():
        s = chunk.loc[(hour >= a) & (hour < e), "stop_id"].value_counts()
        for k, v in s.items():
            counts[b][k] = counts[b].get(k, 0) + int(v)
    print(f"\rrows read: {rows_read:,}", end="")
dep = pd.DataFrame(counts).fillna(0).astype(int)
dep.index.name = "stop_id"
dep = dep.reset_index()
dep.describe()

rows read: 52,609,308

,morning,off_peak,evening
count,249320.000000,249320.000000,249320.000000
mean,11.012305,22.710601,10.797609
std,16.588016,33.245858,16.494746
min,0.000000,0.000000,0.000000
25%,2.000000,6.000000,2.000000
50%,6.000000,12.000000,6.000000
75%,13.000000,26.000000,12.000000
max,328.000000,639.000000,323.000000


## 3. Stops to LSOAs
GTFS `stops.txt` gives coordinates; match each stop to an England LSOA.

In [16]:
stops = pd.read_csv(z.open("stops.txt"), usecols=["stop_id", "stop_lat", "stop_lon"], dtype={"stop_id": str})
stops = stops.merge(dep, on="stop_id", how="inner")
sg = gpd.GeoDataFrame(stops, geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat), crs=4326).to_crs(27700)
lsoa = gpd.read_file(RAW / "lsoa_boundaries_2021_bsc.geojson")
lsoa = lsoa[lsoa["LSOA21CD"].str.startswith("E")].to_crs(27700)[["LSOA21CD", "geometry"]]
j = gpd.sjoin(sg, lsoa, how="inner", predicate="within")
print(f"stops with departures: {len(sg):,}; inside England LSOAs: {len(j):,}")

stops with departures: 249,320; inside England LSOAs: 247,263


## 4. Departures per hour per LSOA

In [17]:
hours = {b: e - a for b, (a, e) in BANDS.items()}
f = j.groupby("LSOA21CD")[list(BANDS)].sum()
for b, h in hours.items():
    f[f"freq_{b}"] = f[b] / h          # departures per hour, all stops in the LSOA
f = f[[f"freq_{b}" for b in BANDS]].reset_index()
base = pd.read_parquet(CLEAN / "lsoa_base.parquet")[["LSOA21CD"]]
f = base.merge(f, on="LSOA21CD", how="left").fillna(0)     # LSOAs with no service = 0
f.describe().T

,count,mean,std,min,25%,50%,75%,max
freq_morning,33755.0,27.018180,43.048350,0.0,6.333333,15.333333,32.000000,2064.000000
freq_off_peak,33755.0,27.856540,43.434268,0.0,6.500000,16.000000,33.166667,1947.333333
freq_evening,33755.0,26.488471,42.805066,0.0,6.000000,15.000000,31.333333,2056.666667


## 5. Checks and save

In [18]:
print("LSOAs with no morning service:", (f.freq_morning == 0).sum(), "of", len(f))
print(f[[c for c in f if c.startswith("freq_")]].sum().round(0))
f.to_parquet(CLEAN / "lsoa_frequency.parquet", index=False)
print("saved")

LSOAs with no morning service: 2510 of 33755
freq_morning     911999.0
freq_off_peak    940298.0
freq_evening     894118.0
dtype: float64
saved
